# Разработка адаптера BLP -> MCLP

In [2]:
import osmnx as ox

ox._version.__version__

'2.0.1'

In [3]:
def load_G_simplyfied():
    g = ox.load_graphml("tests/data/test_rng.ml")
    g = ox.simplify_graph(g)
    return g

G = load_G_simplyfied()

g_nodes = list(G.nodes())

Расчет с использованием чистого алгоритма

In [11]:
from genesis.best_points import BestNodeHillClimbing
from genesis.core import MCLPBase, StateBase, MetricBase, BestPointsBase
from genesis.mclp import Adapter_BLP_MCLP
from genesis.metrics import ArrivalTime
from genesis.states import FirstArrivalUnitState




#====================
class Adapter_BLP_MCLP(MCLPBase):

    def __init__(self,
                 blp_function:    BestPointsBase,
                 state_function:  StateBase      = None,
                 metric_function: MetricBase     = None,
                 **kwargs):
        self.blp_function = blp_function
        super().__init__(state_function, metric_function, **kwargs)
   
    def __call__(self,
                 dynamic_nodes: dict,
                 **kwargs) -> tuple[dict, int | float]:

        # Проверка входящих данных
        if dynamic_nodes is None:
            raise ValueError("Аргумент `dynamic_nodes` должен быть словарем и не может быть равен None!")
        if len(dynamic_nodes) < 1:
            raise ValueError("Аргумент `dynamic_nodes` должен содержать не менее 1 узла!")

        # Получение первого узла из словаря
        start_point, start_key = list(dynamic_nodes.items())[0]

        # Расчет лучшего узла с использованием BLP
        best_node, best_metric = self.blp_function(start_point = start_point, **kwargs)

        # Конвертация результата в результат метода MCLP
        return {best_node: start_key}, best_metric
#====================







start_node = g_nodes[0]
print('Стартовый узел:', start_node)


# Расчет с использованием чистого алгоритма
bnhc = BestNodeHillClimbing(state_function = FirstArrivalUnitState(),
                            metric_function = ArrivalTime())

best_node_bnhc, best_metric_bnhc = bnhc(G, start_point = start_node)
print('Чистый BNHC:', best_node_bnhc, best_metric_bnhc)


# Расет с использованием адаптера
bnhc_mclp = Adapter_BLP_MCLP(blp_function = bnhc)

best_node_bnhc_mclp, best_metric_bnhc_mclp_mclp = bnhc_mclp(env = G, dynamic_nodes = {start_node: 'test'})
print('BNHC(MCLP):', best_node_bnhc_mclp, best_metric_bnhc_mclp_mclp)


Стартовый узел: 290700365
Чистый BNHC: 426923808 7.789429840494647
BNHC(MCLP): {426923808: 'test'} 7.789429840494647


In [55]:
from genesis.best_points import BestNodeBee

# start_node = g_nodes[300]
start_node = g_nodes[200]
print('Стартовый узел:', start_node)

# Расчет с использованием чистого алгоритма
bn = BestNodeBee(state_function = FirstArrivalUnitState(),
                   metric_function = ArrivalTime(),
                   )

best_node_bn, best_metric_bn = bn(G, start_point = start_node)
print('Чистый BNHC:', best_node_bn, best_metric_bn)


# Расет с использованием адаптера
bnmclp = Adapter_BLP_MCLP(blp_function = bn)

best_node_bnmclp, best_metric_bnmclp = bnmclp(env = G, dynamic_nodes = {start_node: 'test'})
print('BN ABC(MCLP):', best_node_bnmclp, best_metric_bnmclp)

Стартовый узел: 1555148537
Чистый BNHC: 426923837 7.6493893689184205
BN ABC(MCLP): {426923837: 'test'} 7.6493893689184205
